In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# R01 — Distributions, moments, and whitening

Read the single primary lesson before this lab. Predict each result. Numerical correctness is not independent mastery. See `representations/SOURCE_AUDIT.md` for provenance.

In [2]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
here = Path.cwd().resolve()
ROOT = next((p for p in [here, *here.parents] if (p / 'src/shape_lab').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook within the extracted course project.')
sys.path.insert(0, str(ROOT / 'src'))
from shape_lab import representations as r
from shape_lab import representation_experiments as exp
print('Course source:', ROOT)


Course source: /tmp/shape-course-q90rdcpz/course


## Predict the covariance before calculating it.

In [3]:
x = np.array([[1.,2.],[3.,4.],[5.,6.]])
print('mean =', x.mean(0))
print('covariance =', r.covariance(x))
assert np.allclose(r.covariance(x), [[4,4],[4,4]])

mean = [3. 4.]
covariance = [[4. 4.]
 [4. 4.]]


## Inspect the recorded real measurement columns and training-only transformation.

In [4]:
result = exp.distribution_lab()
print(json.dumps(result, indent=2))
assert result['features'] == exp.IRIS_FEATURES
assert result['train_after']['covariance_identity_frobenius'] < 1e-8

{
  "observed_source": "bundled Iris; first 100 rows fit, final 50 demonstration only",
  "features": [
    "sepal_length_cm",
    "sepal_width_cm",
    "petal_length_cm",
    "petal_width_cm"
  ],
  "train_before": {
    "n": 100,
    "dimension": 4,
    "mean_norm": 6.952608071795793,
    "mean_coordinate_std": 0.7837846994273638,
    "covariance_identity_frobenius": 2.369614400354564,
    "effective_rank": 1.4495344551560747
  },
  "train_after": {
    "n": 100,
    "dimension": 4,
    "mean_norm": 3.880427132682969e-15,
    "mean_coordinate_std": 0.9999999999999962,
    "covariance_identity_frobenius": 2.6510062659904207e-14,
    "effective_rank": 4.0
  },
  "heldout_after_train_transform": {
    "n": 50,
    "dimension": 4,
    "mean_norm": 2.764227036955555,
    "mean_coordinate_std": 1.3831335751143181,
    "covariance_identity_frobenius": 5.557611488791727,
    "effective_rank": 2.2296241612613676
  },
  "constructed_discrete_whitened": {
    "n": 128,
    "dimension": 2,
    "

## Draw the whitened discrete counterexample. Identity covariance does not create a continuous density.

In [5]:
atoms = np.tile([[-1.,-1.],[-1.,1.],[1.,-1.],[1.,1.]], (32,1))
mu, W = r.fit_whitener(atoms)
z = (atoms-mu) @ W
plt.figure(figsize=(6,5)); plt.scatter(z[:,0], z[:,1]); plt.xlabel('Whitened coordinate 1'); plt.ylabel('Whitened coordinate 2'); plt.title('Four atoms remain four atoms after whitening'); plt.tight_layout(); plt.show()
print(r.covariance(z))

[[1.00000000e+00 4.58702434e-19]
 [4.58702434e-19 1.00000000e+00]]


/tmp/ipykernel_6811/1524137381.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.figure(figsize=(6,5)); plt.scatter(z[:,0], z[:,1]); plt.xlabel('Whitened coordinate 1'); plt.ylabel('Whitened coordinate 2'); plt.title('Four atoms remain four atoms after whitening'); plt.tight_layout(); plt.show()


## Intentionally request an impossible full-rank transformation. Catch only the expected error.

In [6]:
try:
    r.fit_whitener(np.ones((10,4)))
except ValueError as error:
    print('Expected rejection:', error)
else:
    raise AssertionError('Rank deficiency was silently accepted.')

Expected rejection: Covariance is rank deficient or ill-conditioned; choose a lower-dimensional model explicitly.


## Your evidence
Record: observation unit; assumptions; prediction; calculated result; difference from your prediction; and one unsupported conclusion. Attempt the separate learner notebook before opening the reference solutions.